In [ ]:
# 1. REMOVE EVERYTHING TO CLEAR THE CACHE
!pip uninstall -y unsloth unsloth_zoo xformers trl peft accelerate bitsandbytes

# 2. FORCE INSTALL LATEST SYNCED VERSIONS DIRECTLY FROM GITHUB
!pip install --no-cache-dir "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

# 3. INSTALL SUPPORTING LIBRARIES
!pip install --no-cache-dir xformers trl peft accelerate bitsandbytes

# 4. RESTART THE SESSION AUTOMATICALLY
import os
os.kill(os.getpid(), 9)

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ltzej34o/unsloth_292c49d2ad614376ac951c346fe85e10
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ltzej34o/unsloth_292c49d2ad614376ac951c346fe85e10
  Resolved https://github.com/unslothai/unsloth.git to commit 726abd5e6b1888dd5bfb1c47b0b1f9e94b51d5f9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.4.8-py3-none-any.whl size=32234342 sha256=39faf5b2e8cb0d44ea08662a6b4322f1dfef3cbb22b475ed9028ac9eded903d2
  Stored in directory: /tmp/pip-ephem-wheel-cache-qsu2ukql/wheels/60/3e/1f/e576c07051d90cf

In [1]:
# 1. IMPORTS (UNSLOTH MUST BE FIRST)
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import os

# 2. CONFIGURATION
max_seq_length = 2048
load_in_4bit = True

# We use the 3B model to ensure it fits comfortably on the T4 GPU without errors
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
    device_map = {"": 0}, # Forces everything onto the GPU
)

# 3. ADD LoRA ADAPTERS
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. DATA PREPARATION
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = [f"### Question:\n{i}\n\n### Reasoning:\n{o} " for i, o in zip(instructions, outputs)]
    return { "text" : texts, }

# Ensure the file is there
if not os.path.exists("verified_cot_train.jsonl"):
    print("Please upload 'verified_cot_train.jsonl' to the folder on the left!")
else:
    dataset = load_dataset("json", data_files="verified_cot_train.jsonl", split="train")
    dataset = dataset.map(formatting_prompts_func, batched = True,)

    # 5. TRAINING (Standardized for 2026 TRL versions)
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        packing = True,
        args = SFTConfig(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            max_steps = 60,
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 1,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = "outputs",
            report_to = "none",
            packing_strategy = "bfd",
        ),
    )

    trainer.train()
    print("Training Complete! The model now knows how to think.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 215 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.799168
2,0.861516
3,0.836851
4,0.774770
5,0.802053
6,0.870196
7,0.679248
8,0.624367
9,0.651534
10,0.575095


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Training Complete! The model now knows how to think.
